In [6]:
import geopandas as gpd
import folium
from branca.colormap import linear
from IPython.display import HTML, display
from pathlib import Path

source_path = Path(r"Data/edge_counts_by_source_deciles.gpkg")
output_dir = Path(r"Plots/Report_figs")
output_dir.mkdir(parents=True, exist_ok=True)
html_path = output_dir / "edge_counts_by_source_deciles_interactive_map.html"

# Folium expects geographic coordinates, so convert from NZTM to WGS84.
gdf = gpd.read_file(source_path).to_crs(4326)
count_columns = [
    ("mobile_pedestrian_count", "Mobile pedestrian"),
    ("mobile_cycle_count", "Mobile cycle"),
    ("mobile_total_count", "Mobile total"),
    ("strava_pedestrian_count", "Strava pedestrian"),
    ("strava_cycle_count", "Strava cycle"),
    ("strava_total_count", "Strava total"),
    ("wikiloc_pedestrian_count", "Wikiloc pedestrian"),
    ("wikiloc_cycle_count", "Wikiloc cycle"),
    ("wikiloc_total_count", "Wikiloc total"),
    ("alltrails_pedestrian_count", "AllTrails pedestrian"),
    ("alltrails_cycle_count", "AllTrails cycle"),
    ("alltrails_total_count", "AllTrails total"),
    ("total_pedestrian_count", "Overall pedestrian"),
    ("total_cycle_count", "Overall cycle"),
    ("total_count", "Overall total"),
]

def make_style_fn(value_column, colormap):
    def style_fn(feature):
        value = feature["properties"].get(value_column)
        if value is None:
            return {"color": "#9e9e9e", "weight": 2, "opacity": 0.55}
        return {"color": colormap(value), "weight": 3, "opacity": 0.85}

    return style_fn

minx, miny, maxx, maxy = gdf.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="OpenStreetMap",
    control_scale=True,
)
m.fit_bounds([[miny, minx], [maxy, maxx]])

tooltip_fields = ["edge_id", "RDBPT_Sub", "RDBPT_Num"]

for value_column, layer_name in count_columns:
    min_value = float(gdf[value_column].min())
    max_value = float(gdf[value_column].max())
    colormap = linear.YlOrRd_09.scale(min_value, max_value)

    # Sort ascending so higher-decile edges are drawn last (on top).
    layer_gdf = gdf.sort_values(value_column, ascending=True, na_position="first", kind="mergesort")

    layer = folium.FeatureGroup(name=layer_name, show=value_column == "total_count")
    geojson = folium.GeoJson(
        layer_gdf,
        style_function=make_style_fn(value_column, colormap),
        highlight_function=lambda feature: {
            "weight": 6,
            "opacity": 1.0,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=tooltip_fields + [value_column],
            aliases=["Edge ID", "RDBPT Sub", "RDBPT Num", layer_name],
            localize=True,
            sticky=False,
        ),
    )
    geojson.add_to(layer)
    layer.add_to(m)

legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    z-index: 9999;
    background-color: rgba(255, 255, 255, 0.92);
    border: 1px solid #999;
    border-radius: 6px;
    padding: 10px 12px;
    font-size: 12px;
    line-height: 1.2;
    box-shadow: 0 1px 4px rgba(0, 0, 0, 0.2);
">
    <div style="font-weight: 700; margin-bottom: 6px;">
        Track Segment Popularity by Decile
    </div>
    <div style="
        width: 180px;
        height: 12px;
        background: linear-gradient(to right, #ffffcc, #800026);
        border: 1px solid #999;
        margin-bottom: 4px;
    "></div>
    <div style="display: flex; justify-content: space-between; width: 180px;">
        <span>Low</span>
        <span>High</span>
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

m.save(str(html_path))

# Render a clickable link in the notebook and print the saved path for sharing.
display(HTML(f'<a href="{html_path.resolve().as_uri()}" target="_blank">Open interactive HTML map</a>'))
print(f"Saved interactive map to: {html_path.resolve()}")

Saved interactive map to: C:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\Plots\Report_figs\edge_counts_by_source_deciles_interactive_map.html
